# 10 — Gappy Encoder

**Vignette index:** | [`01` GKMKernel basics](01_basic_kernel_matrix.ipynb) | [`02` Distance metrics & kernels](02_distance_metrics_and_kernels.ipynb) | [`03` SVM with kernel](03_svc_with_kernel.ipynb) | [`04` Clustering](04_clustering_sequences.ipynb) | [`05` Long sequence scoring](05_score_long_sequence.ipynb) | [`06` Weighted (WGKM) kernel](06_weighted_kernel.ipynb) | [`07` Transforms & comparison](07_transform_and_comparison.ipynb) | [`08` Windowed 3D tensors](08_windowed_3d_tensors.ipynb) | [`09` Spectrum encoder & NB](09_spectrum_encoder_and_differential.ipynb) | `**10**` Gappy encoder | [`11` Mismatch encoder](11_mismatch_encoder.ipynb) | [`12` Shuffler & chunker](12_shuffler_and_chunker.ipynb)

The `GappyEncoder` counts gappy k-mers — patterns with don't-care positions, e.g. `*--*` matches `AC.G`, `GT.A`, etc.

Two construction modes:
- **Explicit masks:** `masks=["*--*", "*-*--*"]`
- **Gap range:** `L=6, g_min=2, g_max=3`

In [1]:
from kmer.encoders import GappyEncoder
import numpy as np

## 1. Explicit mask with palindromic k-mer demonstration

The sequence `CGGCCGGCG` with mask `*--*` (concrete positions 0 and 3) produces gappy k-mers: `C--G`, `G--G`, `G--C`, `C--C`, `G--G`, `C--C`, `G--G`. With `canonical_rc=True`, these collapse: `C--G` and `G--C` are RC pairs; `G--G` and `C--C` are RC pairs; `G--G` is its own RC (palindrome), so it gets counted twice.

In [2]:
seq = 'CGGCCGGCG'
enc = GappyEncoder(masks=['*--*'])
X = enc.fit_transform([seq])
print('Without canonical_rc:')
print('  Shape:', X.shape)
print('  Total count:', X.sum())

enc_rc = GappyEncoder(masks=['*--*'], canonical_rc=True)
X_rc = enc_rc.fit_transform([seq])
print('With canonical_rc=True:')
print('  Shape:', X_rc.shape)
print('  Total count:', X_rc.sum())
print('  Non-zero columns:', X_rc.nnz)

Without canonical_rc:
  Shape: (1, 16)
  Total count: 6
With canonical_rc=True:
  Shape: (1, 10)
  Total count: 6
  Non-zero columns: 3


## 2. Multiple masks

In [3]:
enc = GappyEncoder(masks=['*--*', '*---*', '*-*--*'])
X = enc.fit_transform(['ACGTACGTACGT'])
print('Shape:', X.shape)
print('Masks:', enc.masks_)

Shape: (1, 96)
Masks: ['*--*', '*---*', '*-*--*']


## 3. Gap range

In [4]:
enc = GappyEncoder(L=5, g_min=2, g_max=3)
X = enc.fit_transform(['ACGTACGTACGT'])
print(f'n_masks: {len(enc.masks_)} (C(5,2)+C(5,3) = {10+10})')
print('Masks:', enc.masks_)
print('Shape:', X.shape)

n_masks: 4 (C(5,2)+C(5,3) = 20)
Masks: ['*--**', '*-*-*', '**--*', '*---*']
Shape: (1, 208)
